# WaferAI — Inference on Colab

Loads your trained `wafer_classifier.h5` and runs the full inference pipeline (preprocess → predict → Grad-CAM → confidence chart) on Colab. Use this if your local Python is 3.13 (no TensorFlow wheel) and you want to test the trained model without setting up a 3.11 venv.

**Prerequisites**
1. Run `train_model.ipynb` first and save `wafer_classifier.h5` to your Google Drive root.
2. Optional: T4 GPU (not required — inference works on CPU in <1 s).
3. Run all cells. Upload an image when prompted.


## 1. Mount Drive and load the model

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

MODEL_PATH = "/content/drive/MyDrive/wafer_classifier.h5"


In [ ]:
import tensorflow as tf
print("TF version:", tf.__version__)

model = tf.keras.models.load_model(MODEL_PATH)
print("Loaded.")
model.summary()


## 2. Upload a wafer-map image

Click **Choose Files** and pick a PNG/JPG. You can also use one of the sample images from `data/sample_wafers/` in the repo.


In [ ]:
from google.colab import files
import io
from PIL import Image
import numpy as np

uploaded = files.upload()
filename = next(iter(uploaded))
original = Image.open(io.BytesIO(uploaded[filename])).convert("RGB")
print(f"Loaded {filename}, size {original.size}")
original


## 3. Preprocess and predict

In [ ]:
IMG_SIZE = 96
CLASS_NAMES = [
    "Center", "Donut", "Edge-Loc", "Edge-Ring", "Loc",
    "Random", "Scratch", "Near-full", "none",
]

resized = original.resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
arr = np.asarray(resized, dtype=np.uint8)
batch = np.expand_dims(arr, 0)

probs = model.predict(batch, verbose=0)[0]
top_idx = int(np.argmax(probs))
print(f"Prediction: {CLASS_NAMES[top_idx]}  ({probs[top_idx]*100:.1f}%)")
print()
for i in np.argsort(probs)[::-1]:
    print(f"  {CLASS_NAMES[i]:<10}  {probs[i]*100:5.1f}%")


## 4. Grad-CAM

Compute the gradient of the predicted class with respect to the last convolutional layer of EfficientNetB0, then weight the feature maps to get a class-discriminative heatmap.


In [ ]:
import cv2

# The model wraps EfficientNetB0 as a single Layer. Pull the inner base and
# pick its last conv layer ('top_conv').
base = next(l for l in model.layers if isinstance(l, tf.keras.Model))
LAST_CONV = "top_conv"
print("Grad-CAM target layer:", LAST_CONV, "→", base.get_layer(LAST_CONV).output.shape)

# Strategy: get conv activations and prediction in two forward passes, then
# attribute via gradients of the predicted class wrt the conv tensor. Falls
# back to channel-mean activation if the gradient is disconnected (which can
# happen with deeply nested Keras models).
batch_f = tf.constant(batch, dtype=tf.float32)
conv_extractor = tf.keras.Model(base.input, base.get_layer(LAST_CONV).output)

with tf.GradientTape() as tape:
    rescaled = batch_f / 255.0
    tape.watch(rescaled)
    conv_out = conv_extractor(rescaled, training=False)
    tape.watch(conv_out)
    preds = model(batch_f, training=False)
    target = preds[:, top_idx]

grads = tape.gradient(target, conv_out)
conv_arr = conv_out[0].numpy()
if grads is None:
    print("(Gradient disconnected — using channel-mean activations as the heatmap.)")
    heat = conv_arr.mean(axis=-1)
else:
    weights = tf.reduce_mean(grads, axis=(0, 1, 2)).numpy()
    heat = (conv_arr * weights).sum(axis=-1)

heat = np.maximum(heat, 0)
heat = heat / max(heat.max(), 1e-6)

orig_arr = np.array(original)
heat_resized = cv2.resize(heat, (orig_arr.shape[1], orig_arr.shape[0]))
heat_8u = np.uint8(255 * heat_resized)
colour = cv2.applyColorMap(heat_8u, cv2.COLORMAP_JET)
colour = cv2.cvtColor(colour, cv2.COLOR_BGR2RGB)
overlay = cv2.addWeighted(orig_arr, 0.55, colour, 0.45, 0)
Image.fromarray(overlay)


## 5. Confidence chart

In [ ]:
import matplotlib.pyplot as plt

order = np.argsort(probs)[::-1]
labels = [CLASS_NAMES[i] for i in order]
values = [probs[i] * 100 for i in order]

plt.figure(figsize=(8, 4.5))
bars = plt.barh(labels, values, color="#3F8EFC")
bars[0].set_color("#E8553F")
plt.gca().invert_yaxis()
plt.xlabel("Confidence (%)")
plt.xlim(0, 100)
plt.grid(axis="x", linestyle=":", alpha=0.4)
for bar, v in zip(bars, values):
    plt.text(min(v + 1.5, 95), bar.get_y() + bar.get_height() / 2, f"{v:.1f}%", va="center")
plt.tight_layout()
plt.show()
